In [6]:
from importlib.metadata import version
print("torch version:", version("torch"))

torch version: 2.5.1


In [7]:
import torch
import torch.nn as nn
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]] # step
)
print("输入形状:", inputs.shape)  # (6, 3)


输入形状: torch.Size([6, 3])


In [8]:
query = inputs[1]   # journey
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print("未归一化的注意力分数:", attn_scores_2)

未归一化的注意力分数: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [9]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("注意力权重:", attn_weights_2)
print("权重和:", attn_weights_2.sum().item())

注意力权重: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
权重和: 1.0


In [10]:
context_vec_2 = torch.zeros(query.shape)
print(context_vec_2)
for i, x_i in enumerate(inputs):
    # print(attn_weights_2[i],x_i)
    m = attn_weights_2[i] * x_i
    #print('标量与向量相乘:',m)
    context_vec_2 += m
    # print('相加后:',context_vec_2)
    
print("第2个词的上下文向量:", context_vec_2)

tensor([0., 0., 0.])
第2个词的上下文向量: tensor([0.4419, 0.6515, 0.5683])


In [11]:
attn_scores = inputs @ inputs.T   # 6x6 矩阵，每个元素是点积
print("点积:\n", attn_scores)
attn_weights = torch.softmax(attn_scores, dim=-1)  # 按行归一化
print("按行归一化:\n", attn_weights)

all_context_vecs = attn_weights @ inputs   # 6x3
print("所有上下文向量:\n", all_context_vecs)
print("形状:", all_context_vecs.shape)

点积:
 tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
按行归一化:
 tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
所有上下文向量:
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
形状: torch.Size([6, 3])


In [20]:
d_in = inputs.shape[1]   # 输入维度 = 3
d_out = 2# 输出维度（可任意）

torch.manual_seed(123)
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# 验证：将输入投影到 2 维空间
queries = inputs @ W_query
keys    = inputs @ W_key
values  = inputs @ W_value
print("queries 形状:", queries.shape)  # (6,2)

queries 形状: torch.Size([6, 2])


In [21]:
query_2 = queries[1]
attn_scores_2 = query_2 @ keys.T   # 点积

d_k = keys.shape[-1]   # =2
attn_weights_2 = torch.softmax(attn_scores_2 / (d_k ** 0.5), dim=-1)
context_vec_2 = attn_weights_2 @ values

print("注意力权重:", attn_weights_2)
print("上下文向量:", context_vec_2)

注意力权重: tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
上下文向量: tensor([0.3061, 0.8210])


In [22]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self, x):
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)
        return attn_weights @ values

torch.manual_seed(123)
sa = SelfAttention(d_in, d_out)
out = sa(inputs)
print("输出形状:", out.shape)   # (6,2)

输出形状: torch.Size([6, 2])
